# 04 — Modelling

Baselines first, then tuned models. A gradient-boosted model that cannot beat regularised logistic regression has not earned its complexity, and on tabular churn data with ~5.6k rows that is a real possibility rather than a rhetorical one.

Cross-validation is **group-aware**, so feature-identical rows cannot span a fold boundary. Tuning targets **PR-AUC**, because the positive class is the one we spend money on.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
cfg = load_config()

In [2]:
from src.data.loader import make_dataset
ds = make_dataset(cfg)
print(f'train {ds.n_train:,} / test {ds.n_test:,}')
print(f'churn {ds.y_train.mean():.4f} / {ds.y_test.mean():.4f}')
print(f'deduplicated {ds.n_duplicates_removed}, '
      f'{ds.n_conflicting_labels} conflicting rows retained')

train 5,621 / test 1,406
churn 0.2645 / 0.2646
deduplicated 16, 42 conflicting rows retained


## Train baselines and tuned candidates

This cell runs the randomised searches and takes a couple of minutes.

In [3]:
from src.models.train import (train_all, comparison_table,
                              select_best, paired_comparison)
results = train_all(ds, cfg)

--- baselines (untuned) ---


  dummy_prior                  PR-AUC 0.2645  ROC-AUC 0.5000  recall 0.0000


  logistic_regression_plain    PR-AUC 0.6690  ROC-AUC 0.8519  recall 0.5413

--- tuned models (RandomizedSearchCV, n_iter=30, scoring=average_precision) ---


  logistic_regression          PR-AUC 0.6704 (+/-0.0300)  ROC-AUC 0.8524  [2s]
      best: {'class_weight': None, 'C': 5.0}


  random_forest                PR-AUC 0.6722 (+/-0.0240)  ROC-AUC 0.8510  [72s]
      best: {'n_estimators': 500, 'min_samples_leaf': 15, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': None}


  xgboost                      PR-AUC 0.6753 (+/-0.0252)  ROC-AUC 0.8512  [14s]
      best: {'subsample': 1.0, 'reg_lambda': 0.5, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.02, 'colsample_bytree': 0.6}


In [4]:
comparison_table(results).round(4)

,model,tuned,pr_auc,pr_auc_std,roc_auc,f1,recall,precision,brier,fit_seconds
0,xgboost,True,0.6753,0.0252,0.8512,0.5877,0.5238,0.6709,0.1326,13.5175
1,random_forest,True,0.6722,0.0240,0.8510,0.5774,0.5030,0.6788,0.1330,72.0034
2,logistic_regression,True,0.6704,0.0300,0.8524,0.5978,0.5393,0.6712,0.1324,1.8040
3,logistic_regression_plain,False,0.6690,0.0301,0.8519,0.5990,0.5413,0.6710,0.1325,0.0000
4,dummy_prior,False,0.2645,0.0004,0.5000,0.0000,0.0000,0.0000,0.1946,0.0000


## Does the complex model earn its place?

The candidates are scored on identical folds, so the fold scores are paired and a **paired** test is the right instrument. Comparing means alone invites reading a 0.004 difference as an improvement when the fold-to-fold spread is 0.025.

In [5]:
best = select_best(results)
cmp = paired_comparison(best, results['logistic_regression_plain'])
for k, v in cmp.items():
    print(f'{k:22s} {v}')

metric                 pr_auc
model_a                xgboost
model_b                logistic_regression_plain
mean_a                 0.6753206677737157
mean_b                 0.6690248686231616
difference             0.0062957991505541155
difference_in_sd       0.2251705805733652
ttest_p                0.4689596945382219
wilcoxon_p             0.625
significant_at_05      False


With five folds the test has very low power, so a non-significant result means **the data cannot distinguish them** — not that they are proven equal. That is the useful conclusion when the simpler model is the one at risk of being discarded for no measured gain.

In [6]:
import pandas as pd
pd.DataFrame({name: r.cv_folds['pr_auc'] for name, r in results.items()
              if not name.startswith('dummy')}).round(4)

,logistic_regression_plain,logistic_regression,random_forest,xgboost
0,0.6730,0.6745,0.7018,0.7027
1,0.7096,0.7120,0.6968,0.7006
2,0.6597,0.6602,0.6615,0.6685
3,0.6185,0.6209,0.6373,0.6337
4,0.6842,0.6847,0.6634,0.6712
